In [1]:
# %%
# =============================================================================
# 11_rag_model_comparison.ipynb
# Financial AI Governance — Model Comparison: gpt-4o-mini vs gpt-4o (Option C)
# Kernel : Python (llm_env)
# Input  : data/processed/dataset_final.json
#          vectordb/ (Chroma persistent stores from 02_rag_pipeline.ipynb)
# Output : results/responses/responses_rag_gpt4o.json
#          results/tables/table_model_summary.csv
# Note   : Stratified sample of 90 items (30 per regulation) only.
#          Full 300-item run is cost-prohibitive for gpt-4o.
#          RAG condition only (top-k=3). Baseline gpt-4o-mini reused from 03.
#          Sampling is stratified by regulation to ensure balanced comparison.
# =============================================================================

# %%
# =============================================================================
# Cell 1. Libraries and Environment Setup
# =============================================================================
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Directory paths — identical to 03_llm_inference.ipynb
DATA_DIR     = '../data/processed'
VDB_DIR      = '../vectordb'
RESPONSE_DIR = '../results/responses'
TABLE_DIR    = '../results/tables'

for d in [RESPONSE_DIR, TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

# API setup
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
EMBED_MODEL    = 'text-embedding-3-small'

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

client     = OpenAI(api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL, api_key=OPENAI_API_KEY)

# Model configuration
MODEL_MINI  = 'gpt-4o-mini'   # existing baseline
MODEL_4O    = 'gpt-4o'        # new comparison model
TOP_K       = 3
SAMPLE_PER_REG = 30           # 30 per regulation = 90 total
RANDOM_SEED    = 42

print(f"[INFO] Comparison models : {MODEL_MINI} vs {MODEL_4O}")
print(f"[INFO] Embedding model   : {EMBED_MODEL}")
print(f"[INFO] Top-k             : {TOP_K}")
print(f"[INFO] Sample size       : {SAMPLE_PER_REG} per regulation = "
      f"{SAMPLE_PER_REG * 3} total")


# %%
# =============================================================================
# Cell 2. Load Dataset and Create Stratified Sample
# =============================================================================
with open(os.path.join(DATA_DIR, 'dataset_final.json'), 'r', encoding='utf-8') as f:
    dataset = json.load(f)
df_full = pd.DataFrame(dataset)
print(f"[INFO] Full dataset: {len(df_full)} records")

# Stratified sample: 30 per regulation, balanced by difficulty
# Mirrors the evaluator robustness check design in 08_evaluator_robustness.ipynb
df_sample = (
    df_full
    .groupby('regulation', group_keys=False)
    .apply(lambda g: g.sample(n=SAMPLE_PER_REG, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)

print(f"[INFO] Stratified sample: {len(df_sample)} records")
print(f"\n  Distribution by regulation:")
print(df_sample['regulation'].value_counts().to_string())
print(f"\n  Distribution by difficulty:")
print(df_sample.groupby(['regulation', 'difficulty']).size().to_string())


# %%
# =============================================================================
# Cell 3. Load Vector Stores
# =============================================================================
VDB_CONFIG = {
    'NIST_AI_RMF'    : {'persist_dir': os.path.join(VDB_DIR, 'nist'),          'collection': 'nist_ai_rmf'},
    'KR_AI_BASIC_ACT': {'persist_dir': os.path.join(VDB_DIR, 'kr_aibasicact'), 'collection': 'kr_aibasicact'},
    'EU_AI_ACT'      : {'persist_dir': os.path.join(VDB_DIR, 'eu_aiact'),      'collection': 'eu_aiact'},
}

vector_stores = {}
for reg_key, cfg in VDB_CONFIG.items():
    vector_stores[reg_key] = Chroma(
        collection_name    = cfg['collection'],
        embedding_function = embeddings,
        persist_directory  = cfg['persist_dir'],
    )
    count = vector_stores[reg_key]._collection.count()
    print(f"  [LOAD] {reg_key:20s} | {count} chunks")

print("[INFO] All vector stores loaded.")


# %%
# =============================================================================
# Cell 4. Prompt Templates (identical to 03_llm_inference.ipynb)
# =============================================================================
SYSTEM_PROMPT = """You are an expert AI governance advisor specializing in financial institution AI compliance.
Your role is to support an AI Review Committee at a financial institution by providing accurate,
regulation-grounded answers to governance questions.

When answering:
1. Cite specific regulatory provisions (article numbers, section codes) where applicable.
2. Identify the governance axis: G1 (Accuracy), G2 (Safety), G3 (Transparency), or G4 (Compliance).
3. Flag high-risk scenarios and recommend human oversight where appropriate.
4. If uncertain, state limitations clearly rather than fabricating information.
5. Keep answers concise, structured, and actionable for a compliance committee."""


def build_rag_prompt(question: str, context: str) -> str:
    """Identical to 03_llm_inference.ipynb build_rag_prompt()."""
    return f"""Answer the following AI governance question using the regulatory context provided below.

--- REGULATORY CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}

Provide a structured answer grounded in the regulatory context above.
Cite specific article numbers or section codes from the context where applicable."""


# %%
# =============================================================================
# Cell 5. Retrieval and Inference Functions
# =============================================================================
def retrieve_context(question: str, regulation: str, k: int = TOP_K) -> str:
    """Identical to 03_llm_inference.ipynb retrieve_context()."""
    if regulation not in vector_stores:
        raise ValueError(f"[ERROR] Unknown regulation: {regulation}")
    docs    = vector_stores[regulation].similarity_search(question, k=k)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    return context


def call_llm(system_prompt: str, user_prompt: str,
             model: str = MODEL_MINI,
             temperature: float = 0.0,
             max_tokens: int = 1000) -> dict:
    """Identical to 03_llm_inference.ipynb call_llm()."""
    try:
        res = client.chat.completions.create(
            model       = model,
            temperature = temperature,
            max_tokens  = max_tokens,
            messages    = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ]
        )
        return {
            'response'         : res.choices[0].message.content.strip(),
            'prompt_tokens'    : res.usage.prompt_tokens,
            'completion_tokens': res.usage.completion_tokens,
            'total_tokens'     : res.usage.total_tokens,
        }
    except Exception as e:
        return {
            'response'         : f'[ERROR] {str(e)}',
            'prompt_tokens'    : 0,
            'completion_tokens': 0,
            'total_tokens'     : 0,
        }


# %%
# =============================================================================
# Cell 6. Cost Estimate Before Running gpt-4o
# =============================================================================
# gpt-4o pricing (as of 2025): ~$2.50/1M input, $10.00/1M output tokens
# Estimate based on RAG k=3 avg tokens from 03_llm_inference.ipynb

AVG_PROMPT_TOK_RAG  = 1_400   # avg prompt tokens in RAG condition
AVG_COMPL_TOK_RAG   = 644     # avg completion tokens in RAG condition
N_SAMPLE            = SAMPLE_PER_REG * 3  # 90

est_input_cost  = (AVG_PROMPT_TOK_RAG  * N_SAMPLE / 1_000_000) * 2.50
est_output_cost = (AVG_COMPL_TOK_RAG   * N_SAMPLE / 1_000_000) * 10.00
est_total       = est_input_cost + est_output_cost

print(f"[INFO] gpt-4o Cost Estimate (90 items)")
print(f"  Avg prompt tokens    : ~{AVG_PROMPT_TOK_RAG:,}")
print(f"  Avg completion tokens: ~{AVG_COMPL_TOK_RAG:,}")
print(f"  Est. input cost      : ${est_input_cost:.3f}")
print(f"  Est. output cost     : ${est_output_cost:.3f}")
print(f"  Est. total cost      : ${est_total:.3f}")
print(f"\n  Proceed? If yes, run Cell 7.")


# %%
# =============================================================================
# Cell 7. Run RAG Inference — gpt-4o (90-item sample)
# =============================================================================
print(f"[RUN] RAG inference — {MODEL_4O} (sample n={len(df_sample)})")
print(f"      Temperature: 0.0 | Max tokens: 1000 | Top-k: {TOP_K}\n")

results_4o   = []
total_tokens = 0

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample),
                   desc=f'RAG {MODEL_4O}'):
    context     = retrieve_context(row['question'], row['regulation'])
    user_prompt = build_rag_prompt(row['question'], context)
    result      = call_llm(SYSTEM_PROMPT, user_prompt, model=MODEL_4O)

    # Field structure identical to 03_llm_inference.ipynb
    results_4o.append({
        'id'               : row['id'],
        'scenario_id'      : row['scenario_id'],
        'regulation'       : row['regulation'],
        'function'         : row['function'],
        'difficulty'       : row['difficulty'],
        'financial_domain' : row['financial_domain'],
        'risk_level'       : row['risk_level'],
        'governance_axis'  : row['governance_axis'],
        'question'         : row['question'],
        'ground_truth'     : row['ground_truth'],
        'legal_basis'      : row['legal_basis'],
        'condition'        : 'rag_gpt4o',
        'top_k'            : TOP_K,
        'context_used'     : context,
        'response'         : result['response'],
        'prompt_tokens'    : result['prompt_tokens'],
        'completion_tokens': result['completion_tokens'],
        'total_tokens'     : result['total_tokens'],
        'model'            : MODEL_4O,
        'temperature'      : 0.0,
    })

    total_tokens += result['total_tokens']
    time.sleep(0.5)  # conservative rate limit for gpt-4o

    idx = len(results_4o)
    if idx % 30 == 0:
        errors  = sum(1 for r in results_4o if r['response'].startswith('[ERROR]'))
        avg_len = sum(len(r['response']) for r in results_4o) / idx
        print(f"  [Checkpoint {idx:2d}/{len(df_sample)}] errors: {errors} | "
              f"avg response: {avg_len:.0f} chars | "
              f"tokens so far: {total_tokens:,}")

# Save
out_path = os.path.join(RESPONSE_DIR, 'responses_rag_gpt4o.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results_4o, f, ensure_ascii=False, indent=2)

errors = sum(1 for r in results_4o if r['response'].startswith('[ERROR]'))
print(f"\n[SAVE] responses_rag_gpt4o.json")
print(f"[INFO] records: {len(results_4o)} | errors: {errors} | "
      f"total tokens: {total_tokens:,} | "
      f"estimated cost: ${total_tokens * 0.000005:.4f}")


# %%
# =============================================================================
# Cell 8. Extract Matched gpt-4o-mini Sample for Fair Comparison
# =============================================================================
# Load full gpt-4o-mini RAG results and filter to same 90 item IDs

rag_mini_path = os.path.join(RESPONSE_DIR, 'responses_rag.json')
with open(rag_mini_path, 'r', encoding='utf-8') as f:
    rag_mini_all = json.load(f)

# Match by item id
sample_ids = set(df_sample['id'].astype(str).tolist())
rag_mini_sample = [
    r for r in rag_mini_all
    if str(r.get('id', '')) in sample_ids
]

print(f"[INFO] gpt-4o-mini matched sample: {len(rag_mini_sample)} records")
print(f"[INFO] gpt-4o sample             : {len(results_4o)} records")

# Save matched mini sample for aligned evaluation
out_mini = os.path.join(RESPONSE_DIR, 'responses_rag_mini_sample.json')
with open(out_mini, 'w', encoding='utf-8') as f:
    json.dump(rag_mini_sample, f, ensure_ascii=False, indent=2)
print(f"[SAVE] responses_rag_mini_sample.json")


# %%
# =============================================================================
# Cell 9. Summary Comparison — gpt-4o-mini vs gpt-4o
# =============================================================================
comparison = {
    f'rag_{MODEL_MINI} (n=90)': rag_mini_sample,
    f'rag_{MODEL_4O} (n=90)'  : results_4o,
}

rows = []
for label, data in comparison.items():
    if not data:
        continue
    avg_ctx   = sum(len(r.get('context_used', '')) for r in data) / len(data)
    avg_resp  = sum(len(r.get('response', ''))      for r in data) / len(data)
    avg_ptok  = sum(r.get('prompt_tokens', 0)       for r in data) / len(data)
    avg_ctok  = sum(r.get('completion_tokens', 0)   for r in data) / len(data)
    total_tok = sum(r.get('total_tokens', 0)         for r in data)
    errors    = sum(1 for r in data if r.get('response','').startswith('[ERROR]'))
    rows.append({
        'Condition'             : label,
        'N'                     : len(data),
        'Avg Context (chars)'   : round(avg_ctx),
        'Avg Response (chars)'  : round(avg_resp),
        'Avg Prompt Tokens'     : round(avg_ptok),
        'Avg Completion Tokens' : round(avg_ctok),
        'Total Tokens'          : total_tok,
        'Errors'                : errors,
    })

df_summary = pd.DataFrame(rows)
print("\n[Table] Model Comparison Summary (RAG, n=90 each)")
print(df_summary.to_string(index=False))

out_tbl = os.path.join(TABLE_DIR, 'table_model_summary.csv')
df_summary.to_csv(out_tbl, index=False, encoding='utf-8-sig')
print(f"\n[SAVE] table_model_summary.csv")

print(f"\n✅ Notebook 11 complete — Next: 12_rag_reranking.ipynb")
print(f"   Then run 04_evaluation_g1_g4.ipynb on:")
print(f"     - responses_rag_gpt4o.json")
print(f"     - responses_rag_mini_sample.json  (matched comparison)")

[INFO] Comparison models : gpt-4o-mini vs gpt-4o
[INFO] Embedding model   : text-embedding-3-small
[INFO] Top-k             : 3
[INFO] Sample size       : 30 per regulation = 90 total
[INFO] Full dataset: 300 records
[INFO] Stratified sample: 90 records

  Distribution by regulation:
regulation
EU_AI_ACT          30
KR_AI_BASIC_ACT    30
NIST_AI_RMF        30

  Distribution by difficulty:
regulation       difficulty  
EU_AI_ACT        advanced         9
                 basic            6
                 intermediate    15
KR_AI_BASIC_ACT  advanced        10
                 basic            8
                 intermediate    12
NIST_AI_RMF      advanced        11
                 basic            3
                 intermediate    16
  [LOAD] NIST_AI_RMF          | 11 chunks
  [LOAD] KR_AI_BASIC_ACT      | 17 chunks
  [LOAD] EU_AI_ACT            | 23 chunks
[INFO] All vector stores loaded.
[INFO] gpt-4o Cost Estimate (90 items)
  Avg prompt tokens    : ~1,400
  Avg completion tokens

RAG gpt-4o:  33%|███████████████████████▎                                              | 30/90 [02:21<04:37,  4.63s/it]

  [Checkpoint 30/90] errors: 0 | avg response: 2418 chars | tokens so far: 67,151


RAG gpt-4o:  67%|██████████████████████████████████████████████▋                       | 60/90 [04:41<02:20,  4.69s/it]

  [Checkpoint 60/90] errors: 0 | avg response: 2478 chars | tokens so far: 125,280


RAG gpt-4o: 100%|██████████████████████████████████████████████████████████████████████| 90/90 [07:14<00:00,  4.83s/it]

  [Checkpoint 90/90] errors: 0 | avg response: 2601 chars | tokens so far: 184,800

[SAVE] responses_rag_gpt4o.json
[INFO] records: 90 | errors: 0 | total tokens: 184,800 | estimated cost: $0.9240
[INFO] gpt-4o-mini matched sample: 90 records
[INFO] gpt-4o sample             : 90 records
[SAVE] responses_rag_mini_sample.json

[Table] Model Comparison Summary (RAG, n=90 each)
             Condition  N  Avg Context (chars)  Avg Response (chars)  Avg Prompt Tokens  Avg Completion Tokens  Total Tokens  Errors
rag_gpt-4o-mini (n=90) 90                 6471                  2691               1547                    512        185378       0
     rag_gpt-4o (n=90) 90                 6471                  2601               1547                    506        184800       0

[SAVE] table_model_summary.csv

✅ Notebook 11 complete — Next: 12_rag_reranking.ipynb
   Then run 04_evaluation_g1_g4.ipynb on:
     - responses_rag_gpt4o.json
     - responses_rag_mini_sample.json  (matched comparison)
